<a href="https://colab.research.google.com/github/wyattae/cosc-650-applied-llm-systems/blob/WE_7/src/week3/week3_prompt_engineering_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 (starter): Prompts as Engineering Artifacts

Runs without an API key: the semantic metric is local, and the model calls fall back to clearly-labeled fixtures so you can see the harness work. Set `GEMINI_API_KEY` to run the prompts for real. Cells marked **TODO (you)** are yours.

Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [1]:
import os, json, pathlib
def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = os.environ.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(api_key=key, base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return client.chat.completions.create(model=model, messages=messages, **kw).choices[0].message.content

LIVE = os.environ.get('GEMINI_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [12]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return str(a).strip().lower() == str(b).strip().lower()
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4146.12it/s]


metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite
**TODO (you):** Choose a complex reasoning task (code review, triage, classification, financial or legal analysis). Author a prompt with a system message, at least three few-shot examples, and chain-of-thought scaffolding that asks for a brief written rationale, not hidden reasoning. Store each prompt version as its own file under version control.

In [13]:
from pathlib import Path

if LIVE: 
  PROMPTS_DIR = Path("prompts")
  PROMPT_V1 = (PROMPTS_DIR / "part1_prompt_v1.txt").read_text()
  PROMPT_V2 = (PROMPTS_DIR / "part1_prompt_v2.txt").read_text()
else:
  PROMPT_V1 = 'Classify the ticket into one of: billing, technical, account, shipping. Return JSON {category, rationale}.'
  PROMPT_V2 = (PROMPT_V1 + ' Classify by the primary intent, not incidental words: if money is only context for a delivery problem, choose shipping.')


## Part 2: Build the test suite
**TODO (you):** Write at least ten input and expected-output pairs. Score each response with two metrics: one exact-match check on the structured field, and one semantic-similarity score (sentence-transformers, run locally on CPU).

In [15]:
if LIVE:
    tests = [
        {'id':1,'ticket':'The linebacker made an aggressive play on the quarterback that play.','cat':'football','why':'Linebacker and quarterback are positions in football.'},
        {'id':2,'ticket':'Hes at the 50, 30, 20, 10, touchdown!','cat':'football','why':'Touchdown is a term used in football.'},
        {'id':3,'ticket':'There defense held them on 4th down for a turnover!','cat':'football','why':'Defense, 4th down and turnover are all terms in football.'},
        {'id':4,'ticket':'Thats a double play to end the inning!','cat':'baseball','why':'Double play and inning are terms in baseball.'},
        {'id':5,'ticket':'This pitcher has been hot all day with his fastball.','cat':'baseball','why':'Pitcher and fastball are terms used in baseball.'},
        {'id':6,'ticket':'The center fielder just robbed that homerun.','cat':'baseball','why':'Centerfielder and homerun are terms used in  baseball.'},
        {'id':7,'ticket':'The blue team won in the overtime shootout!','cat':'hockey','why':'Overtime and shootout are terms used in hockey.'},
        {'id':8,'ticket':'This team from Canada has great puck handling skill and scoring.','cat':'hockey','why':'The terms puck, handling skill and scoring are used in hockey.'},
        {'id':9,'ticket':'That is a double fault from player 1, point goes to player 2!','cat':'tennis','why':'Double fault is a term used in tennis.'},
        {'id':10,'ticket':'He just smashed his racquet after missing that serve!','cat':'tennis','why':'Racquet and serve are terms used in tennis.'},
    ]
else:
    tests = [
        {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
        {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
        {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
        {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
        {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
        {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
        {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
        {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
        {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
        {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
    ]
    FIX = {
        'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
        'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
    }

print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


In [16]:
import time

def run_case(version, prompt, t):
    if LIVE:
        txt = gemini_chat([{'role':'user','content': prompt + '\nTicket: ' + t['ticket']}])
        try:
            d = json.loads(txt); return d.get('category',''), d.get('rationale','')
        except Exception:
            return '', txt or ''
    return FIX[version][t['id']]

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],
                     'expected':t['cat'],
                     'got':cat,
                     'exact':exact_match(t['cat'],cat),
                     'expected_rationale':t['why'],
                     'model_rationale':why,
                     'sem':semantic_sim(t['why'],why)}
                     )
        if LIVE:
            time.sleep(5)
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

## Part 3: Evaluate and show a tradeoff
**TODO (you):** Run the suite against your current prompt and record the results. Then make one prompt edit and re-run. Show a case the edit improved and a case it regressed, with the metric numbers for both versions. If your edit produces no regression, say so and report the metric change you did see, with a short note on why the prompt held..

In [17]:
acc1, r1 = score('v1', PROMPT_V1)

print("PROMPT VERSION #1")

for row in r1:
    print(f"\nTest {row['id']}")
    print(f"Expected: {row['expected']}")
    print(f"Predicted: {row['got']}")
    print(f"Exact Match: {row['exact']}")
    print(f"Expected Rationale: {row['expected_rationale']}")
    print(f"Model Rationale: {row['model_rationale']}")
    print(f"Semantic Similarity: {row['sem']:.3f}")

print(f"\nOverall Exact-Match Accuracy: {acc1:.0%}")


PROMPT VERSION #1

Test 1
Expected: football
Predicted: football
Exact Match: True
Expected Rationale: Linebacker and quarterback are positions in football.
Model Rationale: The description mentions a linebacker and a quarterback, which are key positions in football.
Semantic Similarity: 0.893

Test 2
Expected: football
Predicted: football
Exact Match: True
Expected Rationale: Touchdown is a term used in football.
Model Rationale: The description mentions yard lines (50, 30, 20, 10) and a touchdown, which are key elements of football.
Semantic Similarity: 0.608

Test 3
Expected: football
Predicted: football
Exact Match: True
Expected Rationale: Defense, 4th down and turnover are all terms in football.
Model Rationale: The description mentions a defense, a 4th down, and a turnover, which are key elements of football.
Semantic Similarity: 0.780

Test 4
Expected: baseball
Predicted: baseball
Exact Match: True
Expected Rationale: Double play and inning are terms in baseball.
Model Rational

## Part 4: Find one failure and explain it
The regression you surfaced in Part 3 is your required failure case. Explain why the edit helped one input and hurt another, and describe how you would resolve the tradeoff rather than trading errors back and forth.

In [ ]:
for t in tests:
    e1 = next(r for r in r1 if r['id']==t['id'])['exact']
    e2 = next(r for r in r2 if r['id']==t['id'])['exact']
    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")
# TODO (you): explain why v2 helped one case and hurt another, and how you would resolve the tradeoff.

## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).